## Import Libraries

In [1]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

## Step 1. Data Acquisition

Fetch the dataset and load it into a DataFrame.

In [3]:
newsgroups = fetch_20newsgroups(subset='all',
                                  remove=('headers', 'footers', 'quotes'))

In [4]:
df = pd.DataFrame({'text': newsgroups.data, 'target': newsgroups.target})

In [5]:
df.head()

,text,target
0,\n\nI am sure some bashers of Pens fans are pr...,10
1,My brother is in the market for a high-perform...,3
2,\n\n\n\n\tFinally you said what you dream abou...,17
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,3
4,1) I have an old Jasmine drive which I cann...,4


In [6]:
print(newsgroups.target_names)

['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [7]:
for i, name in enumerate(newsgroups.target_names):
    print(i, "->", name)

0 -> alt.atheism
1 -> comp.graphics
2 -> comp.os.ms-windows.misc
3 -> comp.sys.ibm.pc.hardware
4 -> comp.sys.mac.hardware
5 -> comp.windows.x
6 -> misc.forsale
7 -> rec.autos
8 -> rec.motorcycles
9 -> rec.sport.baseball
10 -> rec.sport.hockey
11 -> sci.crypt
12 -> sci.electronics
13 -> sci.med
14 -> sci.space
15 -> soc.religion.christian
16 -> talk.politics.guns
17 -> talk.politics.mideast
18 -> talk.politics.misc
19 -> talk.religion.misc


## Step 2. Data Preprocessing

Clean and prepare the text data. This includes lowercasing, removing numbers,
 tokenization, punctuation, short words, stop words, and lemmatization.

In [8]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import string

In [9]:
# Lowercasing
df['text_processed'] = df['text'].apply(lambda x: x.lower())
df['text_processed'].head()

0    \n\ni am sure some bashers of pens fans are pr...
1    my brother is in the market for a high-perform...
2    \n\n\n\n\tfinally you said what you dream abou...
3    \nthink!\n\nit's the scsi card doing the dma t...
4    1)    i have an old jasmine drive which i cann...
Name: text_processed, dtype: str

In [10]:
# Cleaning - removing all numbers from text
df['text_processed'] = df['text_processed'].apply(lambda x: re.sub(r'\d+', '', x))
df['text_processed'].head()

0    \n\ni am sure some bashers of pens fans are pr...
1    my brother is in the market for a high-perform...
2    \n\n\n\n\tfinally you said what you dream abou...
3    \nthink!\n\nit's the scsi card doing the dma t...
4    )    i have an old jasmine drive which i canno...
Name: text_processed, dtype: str

In [12]:
# Remove punctuation
df['text_processed'] = df['text_processed'].apply(lambda x:x.translate(str.maketrans('', '', string.punctuation)))

In [13]:
# Tokenization the text by splitting it into words
df['text_processed'] = df['text_processed'].apply(lambda x:x.split())
df['text_processed'].head()

0    [i, am, sure, some, bashers, of, pens, fans, a...
1    [my, brother, is, in, the, market, for, a, hig...
2    [finally, you, said, what, you, dream, about, ...
3    [think, its, the, scsi, card, doing, the, dma,...
4    [i, have, an, old, jasmine, drive, which, i, c...
Name: text_processed, dtype: object

In [14]:
# Filtering out non-alphabetic tokens and short tokens
df['text_processed'] = df['text_processed'].apply(
    lambda x: [token for token in x if token.isalpha() and len(token) > 1]
)

In [15]:
# Stop-words removal
stop_words = set(stopwords.words('english'))

# Getting the set of English stopwords
df['text_processed'] = df['text_processed'].apply(
    lambda x: [token for token in x if token not in stop_words]
)
df['text_processed'].head()

0    [sure, bashers, pens, fans, pretty, confused, ...
1    [brother, market, highperformance, video, card...
2    [finally, said, dream, mediterranean, new, are...
3    [think, scsi, card, dma, transfers, disks, scs...
4    [old, jasmine, drive, cannot, use, new, system...
Name: text_processed, dtype: object

In [16]:
# Stemming
stemmer = PorterStemmer()
df['text_processed'] = df['text_processed'].apply(lambda tokens: [stemmer.stem(token) for token in tokens])

In [17]:
# examples of stemmer applied
words = ['running','runner','run']
for word in words:
    print(stemmer.stem(word))

run
runner
run


In [18]:
words = ['analyze', 'analysis', 'analyzed']
for word in words:
    print(stemmer.stem(word))


analyz
analysi
analyz


In [21]:
# Joining stemming results
df['text_processed'] = df['text_processed'].apply(lambda x: ' '.join)

## Step 3. Feature Extraction

In this step, we will convert the cleaned text data into a numerical representation. We will apply two methods to see their effects on the final model: CountVectorizer and TfidfVectorizer. The main difference between them lies in the type of feature representation they produce. Finally, we will also split the dataset into training and testing sets, which is crucial for training and evaluating our machine learning model.

### 3.1 CountVectorizer Feature Extraction

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [24]:
# Train-test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(df['text_processed'],
                                                    df['target'], test_size=0.2, random_state=42)

In [25]:
# Display the sizes of training and test data
print("Step 3: Feature Extraction\n")
print(f"Training data size: {len(X_train)}")
print(f"Test data size: {len(X_test)}")

Step 3: Feature Extraction

Training data size: 15076
Test data size: 3770
